In [ ]:
import pandas as pd
df = pd.read_parquet("../../data/full.parquet")
df["primary_label"] = df["judge_labels"].apply(lambda ls: ls[0] if ls else "normal")
df.head()

In [ ]:
import matplotlib.pyplot as plt
for metric in ["repetition_ratio", "entropy", "length"]:
    fig, ax = plt.subplots(figsize=(8, 5))
    for (model, mode), g in df.groupby(["model", "mode"]):
        gg = g.groupby("n")[metric].mean()
        ax.plot(gg.index, gg.values, marker="o", label=f"{model}/{mode}")
    ax.set_xscale("log"); ax.set_xlabel("N (log)"); ax.set_ylabel(metric)
    ax.legend(fontsize=7); ax.set_title(f"{metric} vs N")
    fig.savefig(f"../../data/plot_{metric}_vs_n.png", dpi=140, bbox_inches="tight")

In [ ]:
import numpy as np
df["broke"] = df["primary_label"].ne("normal")
pivot = df.pivot_table(index="category", columns="model", values="broke", aggfunc="mean")
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(pivot.values, aspect="auto", cmap="magma", vmin=0, vmax=1)
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns, rotation=45, ha="right")
ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
fig.colorbar(im, label="share non-normal"); fig.savefig("../../data/heatmap_breakdown.png", dpi=140, bbox_inches="tight")

In [ ]:
sub = df[df["model"].isin(["qwen7b-base", "qwen7b-instruct"])]
fig, ax = plt.subplots(figsize=(8, 5))
for model, g in sub.groupby("model"):
    gg = g.groupby("n")["repetition_ratio"].mean()
    ax.plot(gg.index, gg.values, marker="s", label=model)
ax.set_xscale("log"); ax.legend(); ax.set_title("Base vs Instruct: output repetition vs N")
fig.savefig("../../data/base_vs_instruct.png", dpi=140, bbox_inches="tight")